# DC only operation optimization

This notebook is used to prototype the operation optimization of the DC only 
cooling system. It should aim at finding the optimal operation of the DC, that is,
providing the required cooling power at the lowest possible cost (electricity consumption).

minimize: $J\:[u.m.] = \sum_{i=1}^{N} \left( J_{e,i}\:[kW_e] \cdot C_{e,i}\:[u.m./kWh_e] \right) \cdot T_s\:[h]$ 

where:
- $J$: total cost of operation (u.m.)
- $J_{e,i}$: electricity consumption at time $i$ (kW_e)
- $T_s$: sample time (h)

Inputs:
- $T_{amb,i}$: Ambient temperature at time $i$ (°C)
- $C_{e,i}$: Electricity cost at time $i$ (u.m./kWh_e)
- $T_{v,i}$: Desired load vapor temperature at time $i$ (°C)
- $\dot{m}_{v,i}$: Load vapor mass flow rate at time $i$ (kg/s) 

Decision variables:
- $w_{dc}$: DC fan feed frequency (%)
- $q_{c}$: Cooling water recirculation flow rate (m³/h)

Constraints:
- $w_{dc} \in [\underline{w}_{dc}, \overline{w}_{dc}]$ (%)
- $q_{c} \in [\underline{q}_{c}, \overline{q}_{c}]$ (m³/h)


In [1]:
# TODO: Switch using uv python environment to conda in order to use pygmo
# Requirements:
# - [ ] Electricy price timeseries data for a full year (Ce, units?)
# - [ ] Ambient temperature for timeseries for a full year (Tamb, ºC)
# - [ ] Load conditions (mass flow rate and temperature) for a full year (mv and Tv, kg/s and ºC)

# Implementation steps:
# - [ ] Build load conditions profile
# - [ ] If not available, make up some data for electricity price
# - [ ] Define a pygmo problem interface for the DC problem (real-time optimization at each step using some local solver)
# - [ ] Evaluate for a full day


In [3]:
from pathlib import Path
import pandas as pd

import combined_cooler_model
import matlab

from phd_visualizations import save_figure

%load_ext autoreload
%autoreload 2

data_path: Path = Path("../../modeling/assets/data.csv")
cc_model = combined_cooler_model.initialize()

# Load data

df = pd.read_csv(data_path, )
display(df)

results_df: pd.DataFrame
results: list[dict] = []
for idx, ds in df.iterrows():

    Tamb_C = matlab.double([ds["Tamb"]])
    HR_pp = matlab.double([ds["HR"]])
    mv_kgh = matlab.double([ds["mv"]])
    qc_m3h = matlab.double([ds["qc"]])
    Rp = matlab.double([ds["Rp"]])
    Rs = matlab.double([ds["Rs"]])
    wdc = matlab.double([ds["wdc"]])
    wwct = matlab.double([ds["wwct"]])
    Tv = matlab.double([ds["Tv"]])

    # Create the 'options' struct as a Python dictionary
    # options = {
    #     'model_type': 'data',
    #     'parameters': {},  # Empty struct in MATLAB
    #     'x0': float('nan'),  # Optional input as NaN
    #     'lb': matlab.double([43.324458513828800]),
    #     'ub': matlab.double([43.324458513828800]),
    # }

    Ce_kWe, Cw_lh, detailed = cc_model.combined_cooler_model(Tamb_C, HR_pp, mv_kgh, qc_m3h, Rp, Rs, wdc, wwct, Tv, nargout=3)
    results.append(detailed)
    
results_df = pd.DataFrame(results)
display(results_df)

cc_model.terminate()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,date,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,...,Rs,Qreleased,Qabsorbed,Qtransferred_option_1,Qtransferred_option_2,Qtransferred_option_3,Qtransferred_option_4,Qtransferred_option_5,Qtransferred_option_6,Qtransferred_option_7
0,02-Dec-2024,43.324459,20.923333,42.056077,17.998552,30.498067,1.947239,3.261430,42.871410,40.086938,...,0.000,202.144133,198.293522,166.373224,304.024595,178.053259,250.033088,187.296809,136.357530,119.725025
1,02-Dec-2024,42.869820,20.486687,52.055926,17.999154,29.898601,0.595016,2.015121,42.321919,39.623784,...,0.000,206.633807,200.773128,167.234472,303.887404,178.866884,252.533100,189.169538,138.698340,120.854515
2,19-Apr-2024,43.900000,16.440000,51.500000,24.000000,33.190000,0.320000,4.360000,43.260000,40.800000,...,0.954,221.957054,209.201968,172.342256,297.035023,183.690735,220.605447,165.252914,125.739895,119.462572
3,19-Apr-2024,43.640000,16.800000,50.740000,24.000000,33.160000,0.360000,4.760000,42.890000,40.620000,...,0.982,220.683015,205.023972,167.991007,292.655963,179.442931,215.555310,161.469916,122.918057,116.711260
4,10-Jan-2025,43.082367,20.522458,22.689454,23.653935,35.036106,0.170597,5.017707,42.456457,40.196202,...,0.986,146.405638,139.114222,139.165482,275.010397,151.820162,180.974035,135.565495,99.151751,98.170481
5,10-Jan-2025,42.858495,21.204845,22.089150,24.001031,35.037860,0.183182,3.974320,42.342798,40.190653,...,0.505,129.139748,140.432957,133.778931,265.095575,146.085239,172.922655,129.534301,95.266258,94.441571
6,13-Jan-2025,44.360271,12.164853,48.714282,24.002221,37.228958,0.271859,3.893380,43.848984,42.061851,...,0.000,142.427062,132.208377,120.656198,248.032123,132.095507,153.785945,115.199220,80.115999,84.789700
7,13-Jan-2025,44.068646,13.396438,42.654157,23.997975,37.150270,0.130718,4.362098,43.682464,31.364087,...,0.000,133.775166,128.244658,116.725000,242.447284,128.060027,149.187820,111.754819,77.895613,82.220317


> In combined_cooler_model (line 119)
> In combined_cooler_model (line 119)
> In combined_cooler_model (line 119)
> In wct_model (line 52)
In combined_cooler_model (line 112)


,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,qwct_s,qdc,Tdc_in,Tdc_out,Ce_dc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct
0,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,4.204095,109.126308,...,0.000000,11.771053,39.144389,29.990171,1.791859,6.227499,39.144389,28.687123,0.155379,109.126308
1,20.486687,52.055926,310.063340,17.999154,0.631,0.000,34.701531,30.114162,2.852061,163.646362,...,0.000000,6.641688,38.541561,29.543448,0.379688,11.357466,38.541561,28.811256,0.215329,163.646362
2,16.440000,51.500000,333.400000,24.000000,0.747,0.954,16.000000,31.500000,4.980152,182.842851,...,5.792688,6.072000,39.594066,32.775540,0.098359,23.720688,37.928955,32.710932,0.234499,182.842851
3,16.800000,50.740000,331.400000,24.000000,0.365,0.982,25.000000,26.690000,5.016135,133.352228,...,14.965680,15.240000,39.347478,34.285924,0.194095,23.725680,36.154752,32.267193,0.174746,133.352228
4,20.522458,22.689454,219.734942,23.653935,0.009,0.986,11.000000,22.418634,4.649999,129.864926,...,23.112875,23.441050,40.357627,38.071400,0.032940,23.325761,38.092266,34.457942,0.137658,129.864926
5,21.204845,22.089150,193.777819,24.001031,0.009,0.505,11.000000,24.073361,4.830983,140.654673,...,12.011436,23.785022,40.484930,38.249181,0.032940,12.227445,38.288678,30.522068,0.150243,140.654673
6,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,4.920243,0.000000,...,0.000000,23.810203,41.768513,37.119279,0.271859,0.192018,41.768513,41.768513,0.000000,0.000000
7,13.396438,42.654157,200.976603,23.997975,1.000,0.000,0.000000,21.348224,4.777018,157.667067,...,0.000000,0.000000,41.639693,41.639693,0.000000,23.997975,41.639693,36.861620,0.130718,157.667067
